# Methodentest: Zero-Shot-Klassifikation Korpus 1 (AP7)

Setzt `Workflow_Arbeitspakete.md` AP7, Punkte 3+4 um: Testlauf der Zero-Shot-
Klassifikation auf einer Korpus-1-Stichprobe, um zu prüfen, ob Beers 6
Dimensionen mit dieser Methode trennscharf genug klassifizierbar sind, bevor
AP8 (vollständige Analyse) begonnen wird.

**Abweichung vom Dossier-Modellvorschlag:** `PROJEKTDOSSIER.md` schlägt
`deepset/gbert-base` (rein deutsches BERT-Modell, mit dem Zusatz "o.ä.") vor.
Die deskriptive Statistik aus AP4 zeigte aber: Korpus 1 ist zu 66.5% Englisch,
nur 31.8% Deutsch, 1.7% Französisch. Ein rein deutsches Modell würde zwei
Drittel des Korpus schlecht klassifizieren. Entscheid (siehe
`Eymann_Notizen_Methodik.md`): stattdessen ein **mehrsprachiges** Zero-Shot-
Modell verwenden, das Labels und Text in verschiedenen Sprachen versteht —
Label-Sprache wird pro Segment anhand der bereits vorhandenen `sprache`-Spalte
gewählt.

**Modellwahl `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`:** ursprünglich war
`joeddav/xlm-roberta-large-xnli` vorgesehen, dessen SentencePiece-Tokenizer
sich mit aktuellen `transformers`-Versionen (neues Tokenizer-Backend) nicht
laden liess (`protobuf`/`tiktoken`-Konflikt, auch nach Installation von
`protobuf` reproduzierbar). mDeBERTa-v3 ist ein etabliertes, kleineres und
schnelleres mehrsprachiges NLI-Modell mit einem robusteren Tokenizer, das
dieses Problem nicht hat.

In [ ]:
# Imports
import pandas as pd
from transformers import pipeline

PFAD_CLEAN = "../daten/korpus1/Eymann_Korpus1_clean.csv"


## Beers 6 Dimensionen als Hypothesen-Labels (DE/EN/FR)

**Wichtig — Ergebnis des ersten Testlaufs (22.07.2026):** Mit blossen
Einzelwörtern als Labels ("revealing", "smart" etc.) konzentrierten sich 90%
der Stichprobe auf nur 2 der 6 Dimensionen (revealing/enthüllend, accessible/
zugänglich), `prophetic`/`panoramic` kamen kein einziges Mal vor — trotz hoher
Konfidenzwerte (Ø 0.889). Stichprobenkontrolle zeigte unplausible Treffer
(z.B. "Where is my data stored?" → revealing, 0.79). Diagnose: einzelne,
kontextlose Adjektive sind für Zero-Shot-NLI-Modelle zu unspezifisch und
"passen" trivial zu fast jedem Satz mit technischem Vokabular.

**Fix:** volle, an Beers Originalformulierungen angelehnte Hypothesen-Phrasen
statt Einzelwörter (Quellen: Exzerpte zu Beer 2019, Kapitel "The Data Gaze"):

- Speedy: "Speed is the most pressing and dominant feature" (S. 10)
- Accessible: Analytics sind "intuitive and can therefore be easily accessed
  and understood" (S. 10)
- Revealing: Analytics produzieren "objectivity and efficiency ... in the
  production of insights" (S. 12)
- Panoramic: "all-seeing", bietet "panoramic view of the interior and
  exterior world" (S. 12)
- Prophetic: "has both sight and foresight" (S. 14)
- Smart: Algorithmen "take on the thinking", "ready to do the thinking for
  you" (S. 17)

**Zweiter Fund (nach erstem Testlauf mit den Hypothesen-Phrasen oben):**
Konfidenz brach von Ø 0.889 auf Ø 0.582 ein, und ein neues Dominanz-Muster
("prophetisch") trat auf, inkl. klar falscher Treffer mit brauchbarer Konfidenz
(z.B. "Konkrete Kennzahlen zu Nutzung..." → prophetisch, 0.581 — das Gegenteil
von prophetisch) und einem fast bedeutungslosen Ausreisser (0.165), der trotzdem
als "Top"-Treffer gezählt wurde.

**Ursache:** Die Zero-Shot-Pipeline baut aus jedem Label per Default den Satz
`"This example is {label}."`. Mit vollen Sätzen als Label wurde daraus
grammatikalisch kaputter Text wie `"This example is reveals objective insights
and hidden patterns."` — das verwirrt das NLI-Modell statt zu helfen.

**Zweiter Fix:** `hypothesis_template` explizit setzen (`"...stellt die
Technologie als {} dar."` / `"...portrays the technology as {}."`) und Labels
als kurze Adjektiv-Phrasen formulieren, die grammatikalisch in diese Vorlage
passen — das ist die von HuggingFace empfohlene Nutzung der Pipeline.

**Dritter Fund (nach diesem Fix):** Verteilung wurde deutlich plausibler (alle
6 Dimensionen kommen vor, mehrere hochplausible Treffer mit hoher Konfidenz),
aber: (a) ein wiederkehrender Fehlgriff bei kurzen, aufzählungsartigen Sätzen
("Konkrete Kennzahlen zu Nutzung..." → prophetisch, in allen 3 Durchläufen
falsch) und (b) mehrere sehr niedrige "Top"-Scores (0.19, 0.346), bei denen das
Modell eigentlich bei keiner der 6 Dimensionen sicher ist, aber trotzdem einen
Top-Pick liefern musste.

**Dritter Fix — Neutral-Kategorie:** Statt die niedrigen Scores nachträglich
per Schwellenwert abzuschneiden, bekommt das Modell eine 7. Wahlmöglichkeit
("neutral, ohne besonderen technologischen Vorteil"), die es aktiv wählen
kann. Das bildet inhaltlich ab, dass nicht jedes Satz-Segment einer Website
(z.B. FAQ-Fragen, Rechts-/Haftungsklauseln) tatsächlich eine "Envisioning"-
Aussage im Sinne von Beers Data Imaginary ist.

In [ ]:
HYPOTHESEN_VORLAGE = {
    "de": "Diese Aussage stellt die Technologie als {} dar.",
    "en": "This statement portrays the technology as {}.",
    "fr": "Cette déclaration présente la technologie comme {}.",
}

BEER_DIMENSIONEN = {
    "schnell": {
        "de": "schnell und zeitsparend",
        "en": "fast and time-saving",
        "fr": "rapide et permettant de gagner du temps",
    },
    "zugänglich": {
        "de": "zugänglich, weil sie komplexe Analysen intuitiv verständlich macht",
        "en": "accessible, making complex analytics intuitive and easy to understand",
        "fr": "accessible, rendant les analyses complexes intuitives et faciles à comprendre",
    },
    "enthüllend": {
        "de": "enthüllend, weil sie objektive Erkenntnisse und verborgene Muster aufdeckt",
        "en": "revealing, uncovering objective insights and hidden patterns",
        "fr": "révélatrice, dévoilant des insights objectifs et des schémas cachés",
    },
    "panoramisch": {
        "de": "panoramisch, mit einem allumfassenden, allsehenden Überblick über die gesamte Datenlandschaft",
        "en": "panoramic, offering an all-seeing view of the entire data landscape",
        "fr": "panoramique, offrant une vue globale et exhaustive de toutes les données",
    },
    "prophetisch": {
        "de": "prophetisch, weil sie zukünftige Entwicklungen und Ergebnisse vorhersagt",
        "en": "prophetic, predicting future developments and outcomes",
        "fr": "prophétique, prédisant les développements et résultats futurs",
    },
    "smart": {
        "de": "smart, weil lernende Algorithmen das Denken selbst übernehmen",
        "en": "smart, with machine-learning algorithms taking on the thinking itself",
        "fr": "intelligente, des algorithmes d'apprentissage prenant en charge la réflexion elle-même",
    },
    "neutral": {
        "de": "neutral, ohne einen besonderen technologischen Vorteil hervorzuheben",
        "en": "neutral, without highlighting any particular technological advantage",
        "fr": "neutre, sans mettre en avant un avantage technologique particulier",
    },
}

def labels_fuer_sprache(sprache):
    """Gibt {dimension_name: Label-Phrase} für die passende Sprache zurück
    (Default Englisch, falls Sprache nicht DE/EN/FR)."""
    sprache = sprache if sprache in ("de", "en", "fr") else "en"
    return {dim: werte[sprache] for dim, werte in BEER_DIMENSIONEN.items()}

def vorlage_fuer_sprache(sprache):
    sprache = sprache if sprache in ("de", "en", "fr") else "en"
    return HYPOTHESEN_VORLAGE[sprache]


## Stichprobe ziehen

Für den Machbarkeitscheck reicht eine Stichprobe (nicht das volle Korpus —
das ist AP8). Ziel: Mischung aus allen 3 Sprachen und mehreren Anbietern,
nur Segmente mit mind. 5 Wörtern (zu kurze Fragmente sind für Zero-Shot
ohnehin nicht aussagekräftig).

In [ ]:
df = pd.read_csv(PFAD_CLEAN)
df["wortanzahl"] = df["text"].fillna("").apply(lambda t: len(t.split()))
df_kandidaten = df[df["wortanzahl"] >= 5]

# Bewusst ohne groupby().apply(): je nach Pandas-Version wird dabei die
# Gruppierungsspalte aus dem Teil-Dataframe entfernt (KeyError). Stattdessen
# pro Sprache einfach filtern und ziehen.
STICHPROBENGROESSE = 30
anteile = df_kandidaten["sprache"].value_counts(normalize=True)

teilstichproben = []
for sprache, anteil in anteile.items():
    teil = df_kandidaten[df_kandidaten["sprache"] == sprache]
    n = min(len(teil), max(1, round(STICHPROBENGROESSE * anteil)))
    teilstichproben.append(teil.sample(n, random_state=42))

stichprobe = pd.concat(teilstichproben).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Stichprobe: {len(stichprobe)} Segmente")
print(stichprobe["sprache"].value_counts())


## Zero-Shot-Klassifikation (Testlauf)

Modell wird beim ersten Aufruf automatisch heruntergeladen (Internetzugang
nötig, mehrere hundert MB). `multi_label=True`, da eine Aussage durchaus
mehrere Beer-Dimensionen gleichzeitig bedienen kann (z.B. "schnell UND
zugänglich").

In [ ]:
klassifikator = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

ergebnisse = []
for _, row in stichprobe.iterrows():
    labels_dict = labels_fuer_sprache(row["sprache"])  # {dimension_name: hypothesen_phrase}
    label_zu_dimension = {phrase: dim for dim, phrase in labels_dict.items()}
    phrasen = list(labels_dict.values())

    vorlage = vorlage_fuer_sprache(row["sprache"])
    out = klassifikator(row["text"], candidate_labels=phrasen, multi_label=True, hypothesis_template=vorlage)
    bester_index = out["scores"].index(max(out["scores"]))
    bestes_label = out["labels"][bester_index]

    ergebnisse.append({
        "anbieter": row["anbieter"],
        "sprache": row["sprache"],
        "text": row["text"],
        "top_dimension": label_zu_dimension[bestes_label],
        "top_score": round(out["scores"][bester_index], 3),
        "alle_scores": {label_zu_dimension[l]: round(s, 3) for l, s in zip(out["labels"], out["scores"])},
    })

df_ergebnisse = pd.DataFrame(ergebnisse)
df_ergebnisse[["anbieter", "sprache", "text", "top_dimension", "top_score"]]


## Manuelle Prüfung (AP7 Punkt 4)

Stichprobe durchsehen: Wirkt die vom Modell vergebene Top-Dimension inhaltlich
plausibel? Sind die 6 Dimensionen trennscharf genug, oder klassifiziert das
Modell fast alles in 1-2 Dimensionen (Warnsignal für zu unscharfe Kategorien
oder ungeeignetes Modell)? Segmente, bei denen "neutral" gewinnt, sind kein
Fehler, sondern das erwartete Ergebnis für Segmente ohne klaren Bezug zu einer
der 6 Dimensionen (z.B. FAQ, Rechts-/Haftungsklauseln) — diese für die
Häufigkeitsauswertung in AP8 getrennt von den 6 inhaltlichen Dimensionen
behandeln.

**Zweite Sicherheitsebene — Konfidenz-Untergrenze:** Die Neutral-Kategorie
fängt eindeutig unpassende Sätze ab, aber nicht alle (bei manchen kurzen,
generischen Sätzen bleibt der Top-Score sehr niedrig, z.B. 0.19, unabhängig
vom gewählten Label — das Modell ist sich bei keiner der 7 Optionen wirklich
sicher). Segmente unterhalb der Schwelle werden daher zusätzlich als "unklar"
markiert, unabhängig vom technisch gewonnenen Top-Label.

In [ ]:
KONFIDENZ_UNTERGRENZE = 0.3

df_ergebnisse["top_dimension_final"] = df_ergebnisse.apply(
    lambda r: "unklar" if r["top_score"] < KONFIDENZ_UNTERGRENZE else r["top_dimension"],
    axis=1,
)

print("Verteilung der Top-Dimension (vor Konfidenz-Untergrenze):")
print(df_ergebnisse["top_dimension"].value_counts())
print()
print("Verteilung nach Konfidenz-Untergrenze (< 0.3 -> 'unklar'):")
print(df_ergebnisse["top_dimension_final"].value_counts())
print()
print("Durchschnittlicher Top-Score (Konfidenz):", df_ergebnisse["top_score"].mean().round(3))
print()
for _, row in df_ergebnisse.sample(min(10, len(df_ergebnisse)), random_state=1).iterrows():
    print(f"[{row['sprache']}] {row['anbieter']}: \"{row['text'][:100]}\"")
    print(f"   -> {row['top_dimension_final']} ({row['top_score']})")
    print()


## Entscheid festhalten (AP7 Punkt 6)

**Stand nach 4 Testläufen (22.07.2026):** Verteilung über alle 6 Dimensionen
plus Neutral-Kategorie plausibel (keine 1-2-Dimensionen-Dominanz mehr), mehrere
hochplausible Treffer mit hoher Konfidenz. Zwei bekannte Restlimitationen:
(a) vereinzelte hartnäckige Fehlklassifikationen bei kurzen, generischen
Sätzen, (b) einzelne sehr niedrige Konfidenzwerte selbst mit 7 Label-Optionen
— durch die Konfidenz-Untergrenze (< 0.3 → "unklar") als zweite Sicherheits-
ebene abgefangen.

**Entscheid:** ML-Pipeline (Zero-Shot) ist tragfähige Grundlage für AP8 —
mehrsprachiges Modell (`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`), custom
`hypothesis_template`, 7 Labels (6 Beer-Dimensionen + Neutral), Konfidenz-
Untergrenze 0.3. Bekannte Restlimitationen (kurze/generische Sätze teils
unzuverlässig klassifiziert) im Methodikteil offen darlegen, nicht verstecken.

**Noch offen vor AP8:** Erweiterung von Korpus 1 um weitere Seitentypen
(Produktseiten, Blog, Whitepaper) laut ursprünglichem AP3-Plan — aktuell nur
Startseiten (927 Segmente/28 Anbieter). Würde das Korpus substanzieller machen
und vermutlich den Anteil an FAQ-/Rechtstext-Rauschen reduzieren.

Entscheid zusätzlich in `Eymann_Notizen_Methodik.md` und ggf.
Methodenkapitel-Notizen festhalten; nach AP7 Punkt 5 auch mit Dozent Philippe
Saner absprechen.